# ═══════════════════════════════════════════════════════
# STREAMING SERVICE'S LOST EPISODES — DATA CLEANING PIPELINE
# Proofpoint Intern Program 2026
## Bruno Iván Vargas Mancilla
# ═══════════════════════════════════════════════════════

## Introduction

**Context:**
A streaming platform is digitizing its catalog of series, seasons and episodes.
During data ingestion, no validation or uniqueness checks were applied,
resulting in a corrupted catalog with missing data, inconsistent values
and duplicated episodes.

**Goal:**
Process the CSV file and produce a clean, corrected and de-duplicated catalog,
along with a data quality report summarizing all corrections made.

**Pipeline Overview:**
This notebook implements a data cleaning pipeline composed of:
  1. Data Loading
  2. Raw Data Profiling (Parsing & Inspection)
  3. Standardization Layer
  4. Data Validation & Cleaning
  5. Deduplication
  6. Clean Dataset Generation
  7. Data Quality Report

Each stage is isolated to ensure clarity, reproducibility, and traceability.

#### Structure of the dataset (by rules defined in the challenge)

Description of dataset (columns and data types):

1.	**series_name**: SeriesName
- Description: : Name of the series (required)
- Variable type: Categorical nominal variable
- Data type: string
2.	**season_number**: Season Number
- Description: Season number (may be missing or invalid)
- Variable type: Quantitative discrete variable
- Data type: int
3.	**ep_number**: Episode Number
- Description: Episode number (may be missing or invalid)
- Variable type: Quantitative discrete variable
- Data type: int
4.	**ep_title**: Episode Title
- Description: Episode title (may be missing)
- Variable type: Categorical nominal variable
- Data type: string
5.	**air_date**: Air Date
- Description: Date when the episode aired (may be missing or invalid)
- Variable type: Date variable
- Data type: date

### Dataset Note:
Since no real CSV was provided, a synthetic test dataset (input_episodes.csv)was generated to simulate all possible edge cases described in the challenge, containing:
  - ~101 clean, realistic records across 5 series: Breaking Bad, Better Call Saul, The Big Bang Theory, Two and a Half Men, and Dark.
  - ~42 records with intentional errors covering all validation rules described in the challenge, mixed randomly throughout the dataset.

**Error cases include**: Missing fields (as NaN and as empty strings), semantically empty values ("N/A", "none", "--"), invalid formats (non-numeric season/episode numbers, impossible dates, extra whitespace, diacritics), and duplicate entries of various kinds.

The goal is to demonstrate that the cleaning pipeline correctly handles all these cases according to the rules defined in the challenge.

The pipeline is designed to be dataset-agnostic and should work 
with any input following the described schema.



## 1) Data Loading

**Objective:** Import necessary libraries and load the raw CSV file into a Pandas DataFrame without applying any initial transformations.

In [330]:
# Importar las librerías necesarias
import pandas as pd         # data manipulation with tabular structures
import numpy as np          # numerical operations
import re                   # regular expressions for string cleaning
import unicodedata          # normalization of special characters and accents
from datetime import datetime   # date validation

In [331]:
# Define the path to the CSV file
# Note: Make sure that 'input_episodes.csv' is in the same directory as this notebook, or provide the correct path.
file_path = 'input_episodes.csv'

**Note on Data Ingestion:** 
By default, Pandas converts completely empty cells in the CSV into `NaN` (Not a Number), which is the idiomatic representation of missing data in Python. Pandas automatically handles this during the loading process, so we can rely on `NaN` to identify truly empty fields. However, other forms of "missing" data (like "N/A", "none", "--", or strings with only spaces) will be loaded as strings and need to be handled separately in the cleaning process.

In [332]:
# Load raw dataset into memory
df_raw = pd.read_csv(file_path, delimiter=',')

## 2) Raw Data Profiling (Parsing & Inspection)
**Objective:** Inspect the raw dataset to understand its structure, document its initial state, and identify data quality issues before applying any modifications.

**Stages:**
1. Overview & Column Types
2. Missing Values Analysis
3. Invalid Values Detection
4. String Issues Detection
5. Duplicates Detection
6. Summary of Findings

### 2.1) Overview
General inspection of the dataset to understand its structure, dimensions, and basic characteristics.

In [333]:
# Dataset preview
df_raw.head(10)

,SeriesName,SeasonNumber,EpisodeNumber,EpisodeTitle,AirDate
0,Better Call Saul,1,7,Bingo,2015-03-16
1,Bréaking Bäd,1,1,Pilot,2008-01-20
2,Dark,1,2,Lies,2017-12-02
3,Better Call Saul,3,3,Sunk Costs,2017-04-24
4,The Big Bang Theory,1,5,The Hamburger Postulate,2007-10-22
5,Dark,1,2,Lies,Unknown
6,Two and a Half Men,1,2,"Alan Harper, Frontier Psychiatrist",2003-09-29
7,The Big Bang Theory,2,2,The Codpiece Topology,2008-09-29
8,The Big Bang Theory,2,9,The White Asparagus Triangulation,
9,Better Call Saul,1,5,Alpine Shepherd Boy,2015-03-02


In [334]:
# Dataset shape
print(f"Shape: \n Total records: {df_raw.shape[0]} \n Dimension (total columns): {df_raw.shape[1]}")

Shape: 
 Total records: 143 
 Dimension (total columns): 5


### 2.2) Column Types
Analyze the data types of each column and determine the expected types for each field.

In [335]:
# Column names and data types
dtypes_df = pd.DataFrame({
    "Column": df_raw.columns,
    "Data Type": df_raw.dtypes.values
})

dtypes_df

,Column,Data Type
0,SeriesName,object
1,SeasonNumber,object
2,EpisodeNumber,object
3,EpisodeTitle,object
4,AirDate,object


##### Column Types Analysis
At this stage, all columns are loaded as `object` type in Pandas:
- SeriesName → object
- SeasonNumber → object
- EpisodeNumber → object
- EpisodeTitle → object
- AirDate → object

This is expected due to the inconsistent and unvalidated nature of the input dataset.

##### Expected Data Types
Based on the problem definition, each column should ideally have the following type:

- **SeriesName** → string  
  (Required field, used as primary identifier)

- **SeasonNumber** → integer  
  (May contain invalid or missing values, handled by defaulting to 0)

- **EpisodeNumber** → integer  
  (May contain invalid or missing values, handled by defaulting to 0)

- **EpisodeTitle** → string  
  (Optional, defaults to "Untitled Episode" if missing)

- **AirDate** → date (YYYY-MM-DD format) or categorical ("Unknown")  
  (May contain invalid formats or missing values, handled by defaulting to "Unknown")

##### Important Consideration

At this stage (Parsing and Inspection), no type conversion is applied yet.

This is intentional because:

- The dataset contains invalid and inconsistent values
- Converting types prematurely may result in errors or data loss

Type conversion will be handled later during the **Data Cleaning** phase,
after all invalid values have been properly identified and corrected.

### 2.3) Missing Values Analysis
Identify and quantify missing values (NaN, semantically empty strings, whitespace-only strings).

**Notes on Missing Data:**
Missing data in this unvalidated dataset manifests in three distinct ways:
1. **NaN:** True empty cells in the CSV, parsed natively by Pandas.
2. **Semantically Empty Strings:** Values like "N/A", "none", or "--" that represent missing data but are loaded as text.
3. **Whitespace-only Strings:** Cells containing only spaces (e.g., "   "), which appear to have content but are functionally empty.

#### 2.3.1) Total Records

In [336]:
# Count total rows in the dataset (same for all columns, including those with missing values)
total_rows = len(df_raw)

# Count non-null values per column
non_null_counts = df_raw.count().sort_values(ascending=False)
print(f"Total non-null values per column:\n{non_null_counts}")

# DataFrame with three columns: Column name, Total rows, Non-null count
counts_df = pd.DataFrame({
    "Column": df_raw.columns,
    "Total (including nulls)": total_rows,
    "Total Non-null": non_null_counts.values
})

# Display the summary table
counts_df

Total non-null values per column:
SeriesName       141
SeasonNumber     141
EpisodeTitle     141
EpisodeNumber    140
AirDate          138
dtype: int64


,Column,Total (including nulls),Total Non-null
0,SeriesName,143,141
1,SeasonNumber,143,141
2,EpisodeNumber,143,141
3,EpisodeTitle,143,140
4,AirDate,143,138


#### 2.3.2) NaN Values
This are fields that were empty in the original CSV.
Pandas automatically converts them to NaN when reading the file.

In [337]:
# NaN values per column
nan_counts = df_raw.isnull().sum().sort_values(ascending=False)
print(f"\nNaN values per column:\n{nan_counts}")

# Dataframe for better visualization
nan_counts_df = nan_counts.reset_index()
nan_counts_df.columns = ['Column', 'NaN Count']
nan_counts_df


NaN values per column:
AirDate          5
EpisodeNumber    3
SeriesName       2
SeasonNumber     2
EpisodeTitle     2
dtype: int64


,Column,NaN Count
0,AirDate,5
1,EpisodeNumber,3
2,SeriesName,2
3,SeasonNumber,2
4,EpisodeTitle,2


In [338]:
# Visualization of rows with at least one NaN
nan_rows = df_raw[df_raw.isnull().any(axis=1)]
print(f"\nRows with at least one NaN: {len(nan_rows)}")

# Show rows with at least one NaN
nan_rows


Rows with at least one NaN: 11


,SeriesName,SeasonNumber,EpisodeNumber,EpisodeTitle,AirDate
10,Breaking Bad,3,NaN,Sunset,2010-04-25
27,NaN,2,3,Some Episode,2009-03-22
40,Breaking Bad,4,9,NaN,2011-09-04
43,Breaking Bad,5,9,Blood Money,NaN
54,Breaking Bad,NaN,8,Cornered,2011-08-21
62,Better Call Saul,NaN,2,Cobbler,2016-02-22
77,Breaking Bad,1,NaN,NaN,NaN
90,NaN,1,1,Pilot,2008-01-20
105,Better Call Saul,3,9,Fall,NaN
108,Dark,1,--,,NaN


#### 2.3.3) Semantically Empty Values
This are fields that contain values like "N/A", "none", "--", "unknown".    
Technically they are present strings, but they mean "I don't have data".

We define a set of strings that we consider semantically empty, based on common conventions for representing missing data in text form.

Detection is done by comparing each value against the SEMANTIC_EMPTY set after stripping and lowercasing for consistency.

In [339]:
# List of strings considered semantically empty
SEMANTIC_EMPTY = {"n/a", "na", "none", "unknown", "--", "null"}

In [340]:
# Create a mask for each column that identifies rows with semantically empty values (e.g., "N/A", "none", "--", etc.)
# sem_mask → True for rows containing a semantically empty string
sem_mask = df_raw.apply(
    lambda col: col.dropna() # ignore NaN values (already counted in a previous step)
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(SEMANTIC_EMPTY)
)

# Strings semantically empty per column 
sem_empty_counts = sem_mask.sum().sort_values(ascending=False)
print(f"\nSemantically empty strings per column:\n{sem_empty_counts}")

# Convert the counts into a DataFrame for better visualization
sem_empty_counts_df = sem_empty_counts.reset_index()
sem_empty_counts_df.columns = ['Column', 'Sem. Empty Count']
sem_empty_counts_df


Semantically empty strings per column:
AirDate          3
EpisodeNumber    2
EpisodeTitle     2
SeriesName       0
SeasonNumber     0
dtype: object


,Column,Sem. Empty Count
0,AirDate,3
1,EpisodeNumber,2
2,EpisodeTitle,2
3,SeriesName,0
4,SeasonNumber,0


In [341]:
# Visualization of rows with at least one semantically empty value
emp_empty_rows = df_raw[sem_mask.any(axis=1)]
print(f"\nRows with at least one semantically empty value: {len(emp_empty_rows)}")

# Show rows with at least one semantically empty value
emp_empty_rows


Rows with at least one semantically empty value: 7


,SeriesName,SeasonNumber,EpisodeNumber,EpisodeTitle,AirDate
5,Dark,1,2,Lies,Unknown
108,Dark,1,--,,NaN
113,Better Call Saul,2,NaN,none,NaN
116,Better Call Saul,2,9,none,2016-04-11
118,Breaking Bad,1,2,Cat's in the Bag,Unknown
120,Better Call Saul,1,2,Mijo,Unknown
134,Better Call Saul,1,--,Hero,2015-02-23


#### 2.3.4) Whitespace-only Values
These are fields that appear to have content but are actually empty, containing only whitespace (e.g., "   ").

Technically they have content but after stripping they are empty.

Detection is done by stripping the string and checking if the result equals "".

In [342]:
# Create a mask for each column that identifies rows containing only whitespace.
# ws_mask → True for rows where the string consists solely of spaces.
ws_mask = df_raw.apply(
    lambda col: col.dropna() # ignore NaN values (already counted in a previous step)
    .astype(str)
    .str.strip()
    .eq("")
)

# Count how many whitespace-only values exist per column
ws_counts = ws_mask.sum().sort_values(ascending=False)
print(f"\nWhitespace-only strings per column:\n{ws_counts}")

# Convert the counts into a DataFrame for better visualization
whitespace_counts_df = ws_counts.reset_index()
whitespace_counts_df.columns = ['Column', 'Whitespace Only Count']
whitespace_counts_df


Whitespace-only strings per column:
EpisodeTitle     2
SeriesName       1
AirDate          1
SeasonNumber     0
EpisodeNumber    0
dtype: object


,Column,Whitespace Only Count
0,EpisodeTitle,2
1,SeriesName,1
2,AirDate,1
3,SeasonNumber,0
4,EpisodeNumber,0


In [343]:
# Visualization of rows with at least one whitespace-only value
ws_rows = df_raw[ws_mask.any(axis=1)]
print(f"\nRecords with at least one problematic value: {len(ws_rows)}")

# Show rows with at least one whitespace-only value
ws_rows


Records with at least one problematic value: 4


,SeriesName,SeasonNumber,EpisodeNumber,EpisodeTitle,AirDate
8,The Big Bang Theory,2,9,The White Asparagus Triangulation,
70,Dark,1,6,,2017-12-01
108,Dark,1,--,,NaN
117,,1,5,Another One,2010-04-18


#### 2.3.5) Combined Missing Definition

In [344]:
# Calculate the total number of missing values per column
# (sum of NaN, semantically empty strings, and whitespace-only strings)
total_missing_counts = nan_counts + sem_empty_counts + ws_counts
print(f"\nTotal missing values per column (NaN + semantically empty + whitespace-only):\n{total_missing_counts}")

# Convert the counts into a DataFrame for better visualization
total_missing_counts_df = total_missing_counts.reset_index()
total_missing_counts_df.columns = ['Column', 'Total Missing Count']
total_missing_counts_df


Total missing values per column (NaN + semantically empty + whitespace-only):
AirDate          9
EpisodeNumber    5
EpisodeTitle     6
SeasonNumber     2
SeriesName       3
dtype: object


,Column,Total Missing Count
0,AirDate,9
1,EpisodeNumber,5
2,EpisodeTitle,6
3,SeasonNumber,2
4,SeriesName,3


#### 2.3.6) Summary Table

In [345]:
# Create a Series with the total number of rows (same value for all columns)
total_rows_series = pd.Series(total_rows, index=df_raw.columns)

# Concatenate all counts into a single DataFrame
total_data = pd.concat([total_rows_series, non_null_counts, total_missing_counts, nan_counts, sem_empty_counts, ws_counts], axis=1, keys=['Total', 'Total Non-null values', 'Total Missing', 'NaN (null)', 'Sem. Empty', 'Whitespace Only'])

# Set index name for clarity
total_data.index.name = 'Column'
total_data

,Total,Total Non-null values,Total Missing,NaN (null),Sem. Empty,Whitespace Only
Column,,,,,,
SeriesName,143,141,3,2,0,1
SeasonNumber,143,141,2,2,0,0
EpisodeNumber,143,140,5,3,2,0
EpisodeTitle,143,141,6,2,2,2
AirDate,143,138,9,5,3,1


### 2.4) Invalid Values Detection
Identify and quantify values that do not conform to expected formats or types (e.g., non-numeric season/episode numbers, impossible dates).

- Invalid Season/Episode Numbers
- Invalid Dates
- String Issues

#### 2.4.1) Invalid Season/Episode Numbers
Should be non-negative integers. Invalid examples include "3.5", "-1", "one", "--2".

In [346]:
# Function to check if a value is a valid non-negative integer
def is_valid_non_negative_int(x):
    try:
        return int(x) >= 0
    except:
        return False

In [347]:
# Check SeasonNumber for invalid values
invalid_season_mask = ~df_raw['SeasonNumber'].apply(is_valid_non_negative_int)

# Count and print the number of invalid SeasonNumber entries
invalid_season_count = invalid_season_mask.sum()
print("Invalid SeasonNumber:", invalid_season_count)

# Show a preview of rows with invalid SeasonNumber (first 5)
df_raw[invalid_season_mask].head()

Invalid SeasonNumber: 6


,SeriesName,SeasonNumber,EpisodeNumber,EpisodeTitle,AirDate
25,Two and a Half Men,3.5,4,An Episode,2005-10-11
28,The Big Bang Theory,one,10,The Loobenfeld Decay,2007-11-12
30,Dark,--2,7,The White Devil,2019-07-01
31,Dark,-3,3,Ghosts,2019-06-21
54,Breaking Bad,NaN,8,Cornered,2011-08-21


In [348]:
# Check EpisodeNumber for invalid values
invalid_episode_mask = ~df_raw['EpisodeNumber'].apply(is_valid_non_negative_int)

# Count and print the number of invalid EpisodeNumber entries
invalid_episode_count = invalid_episode_mask.sum()
print("Invalid EpisodeNumber:", invalid_episode_count)

# Show a preview of rows with invalid EpisodeNumber (first 5)
df_raw[invalid_episode_mask].head()

Invalid EpisodeNumber: 9


,SeriesName,SeasonNumber,EpisodeNumber,EpisodeTitle,AirDate
10,Breaking Bad,3,NaN,Sunset,2010-04-25
41,Better Call Saul,1,3.5,Nacho,2015-02-16
47,Breaking Bad,2,one,Seven Thirty-Seven,2009-03-08
77,Breaking Bad,1,NaN,NaN,NaN
104,Two and a Half Men,2,-1,Ghosts,2004-10-25


#### 2.4.2) Invalid Dates
Should be in YYYY-MM-DD format and represent valid calendar dates. Invalid examples include "2022-99-40", "not-a-date", "0000-00-00".

In [349]:
# Function to check if a value is a valid date in YYYY-MM-DD format
def is_valid_date(x):
    try:
        datetime.strptime(str(x), "%Y-%m-%d")
        return True
    except:
        return False

In [350]:
# Check AirDate for invalid values
invalid_date_mask = ~df_raw['AirDate'].apply(is_valid_date)

# Count and print the number of invalid AirDate entries
invalid_date_count = invalid_date_mask.sum()
print("Invalid AirDate:", invalid_date_count)

# Show a preview of rows with invalid AirDate (first 5)
df_raw[invalid_date_mask].head()

Invalid AirDate: 12


,SeriesName,SeasonNumber,EpisodeNumber,EpisodeTitle,AirDate
5,Dark,1,2,Lies,Unknown
8,The Big Bang Theory,2,9,The White Asparagus Triangulation,
20,The Big Bang Theory,1,10,The Loobenfeld Decay,not-a-date
33,Better Call Saul,2,10,Klick,2022-99-40
43,Breaking Bad,5,9,Blood Money,NaN


### 2.5) String Issues Detection
Identify and quantify string inconsistencies such as leading/trailing spaces, multiple internal spaces, diacritics, and case variations (Example: "Lost" vs " lost " vs "LOST").

In [351]:
# Detect leading/trailing spaces
string_issues = df_raw['SeriesName'].astype(str)
has_spaces = string_issues.str.strip() != string_issues

print(f"Records with leading/trailing spaces in SeriesName: {has_spaces.sum()}")

# Show a preview of rows with leading/trailing spaces in SeriesName (first 5)
df_raw[has_spaces].head()

Records with leading/trailing spaces in SeriesName: 2


,SeriesName,SeasonNumber,EpisodeNumber,EpisodeTitle,AirDate
14,breaking bad,1,1,Pilot,2008-01-20
117,,1,5,Another One,2010-04-18


In [352]:
# Detect multiple internal spaces (e.g., "The  Big  Bang Theory")
has_multiple_spaces = string_issues.str.contains(r'\s{2,}', regex=True)

print(f"Records with multiple internal spaces in SeriesName: {has_multiple_spaces.sum()}")

# Show a preview of rows with multiple internal spaces in SeriesName (first 5)
df_raw[has_multiple_spaces].head()

Records with multiple internal spaces in SeriesName: 3


,SeriesName,SeasonNumber,EpisodeNumber,EpisodeTitle,AirDate
14,breaking bad,1,1,Pilot,2008-01-20
49,The Big Bang Theory,1,1,Pilot,2007-09-24
117,,1,5,Another One,2010-04-18


### 2.6) Duplicates Detection
In this stage, we are only identifying potential duplicates based on the defined rules of identity, without yet applying any deduplication logic.


#### 2.6.1) Duplicate Detection Based on Exact Row Matches

In [353]:
# Exact duplicate rows (all columns identical)
duplicates_df = df_raw[df_raw.duplicated()]
duplicates_count = duplicates_df.shape[0]
print(f"Number of duplicate rows: {duplicates_count}")
duplicates_df

Number of duplicate rows: 3


,SeriesName,SeasonNumber,EpisodeNumber,EpisodeTitle,AirDate
115,Breaking Bad,1,1,Pilot,2008-01-20
126,Dark,1,1,Secrets,2017-12-01
137,Better Call Saul,1,1,Uno,2015-02-08


#### 2.6.2) Duplicate Detection Based on Identity Rules

In [354]:
# Create a copy specifically for preview/exploratory analysis
df_preview = df_raw.copy()

In [355]:
# Temporary normalization for analysis (non-destructive to df_raw)
df_preview['SeriesName_norm_preview'] = df_preview['SeriesName'].astype(str).str.strip().str.lower()
df_preview['EpisodeTitle_norm_preview'] = df_preview['EpisodeTitle'].astype(str).str.strip().str.lower()

##### 2.6.2.1) Rule 1: (SeriesName_normalized, SeasonNumber, EpisodeNumber)

In [356]:
# Rule 1:
dup_rule_1 = (
    df_preview.groupby(['SeriesName_norm_preview', 'SeasonNumber', 'EpisodeNumber'])
    .size()
    .reset_index(name='count')
    .sort_values(by='count', ascending=False)
)

print(f"Number of potential duplicate groups based on Rule 1: {(dup_rule_1['count'] > 1).sum()}")

# Show the groups of potential duplicates based on Rule 1 (first 5)
dup_rule_1[dup_rule_1['count'] > 1].head()

Number of potential duplicate groups based on Rule 1: 9


,SeriesName_norm_preview,SeasonNumber,EpisodeNumber,count
2,better call saul,1,1,3
27,breaking bad,1,1,3
10,better call saul,2,1,2
28,breaking bad,1,2,2
3,better call saul,1,2,2


##### 2.6.2.2) Rule 2: (SeriesName_normalized, 0, EpisodeNumber, EpisodeTitle_normalized)

In [357]:
# Rule 2:
dup_rule_2 = (
    df_preview.groupby(['SeriesName_norm_preview', 'EpisodeNumber', 'EpisodeTitle_norm_preview'])
    .size()
    .reset_index(name='count')
    .sort_values(by='count', ascending=False)
)

print(f"Number of potential duplicate groups based on Rule 2: {(dup_rule_2['count'] > 1).sum()}")

# Show the groups of potential duplicates based on Rule 2 (first 5)
dup_rule_2[dup_rule_2['count'] > 1].head(10)

Number of potential duplicate groups based on Rule 2: 9


,SeriesName_norm_preview,EpisodeNumber,EpisodeTitle_norm_preview,count
4,better call saul,1,uno,3
31,breaking bad,1,pilot,3
74,dark,1,secrets,2
36,breaking bad,2,cat's in the bag,2
7,better call saul,2,cobbler,2
8,better call saul,2,mijo,2
93,the big bang theory,10,the loobenfeld decay,2
76,dark,2,lies,2
77,dark,3,ghosts,2


##### 2.6.2.3) Rule 3: (SeriesName_normalized, SeasonNumber, 0, EpisodeTitle_normalized)

In [358]:
# Rule 3:
dup_rule_3 = (
    df_preview.groupby(['SeriesName_norm_preview', 'SeasonNumber', 'EpisodeTitle_norm_preview'])
    .size()
    .reset_index(name='count')
    .sort_values(by='count', ascending=False)
)

print(f"Number of potential duplicate groups based on Rule 3: {(dup_rule_3['count'] > 1).sum()}")

# Show the groups of potential duplicates based on Rule 3 (first 5)
dup_rule_3[dup_rule_3['count'] > 1].head(10)

Number of potential duplicate groups based on Rule 3: 13


,SeriesName_norm_preview,SeasonNumber,EpisodeTitle_norm_preview,count
33,breaking bad,1,pilot,3
7,better call saul,1,uno,3
69,dark,1,,2
73,dark,1,secrets,2
36,breaking bad,2,down,2
29,breaking bad,1,cat's in the bag,2
48,breaking bad,3,sunset,2
14,better call saul,2,none,2
39,breaking bad,2,seven thirty-seven,2
6,better call saul,1,nacho,2


**Notes on Normalization:**
- The normalization applied here is only for exploratory purposes and does not modify the original dataset.
- It allows us to approximate the duplicate detection rules defined in the challenge and identify potential duplicate groups.
- The actual normalization and deduplication will be performed in later stages.

### 2.7) Summary of Findings
Based on the exploratory data analysis, the raw dataset exhibits the following issues that must be addressed in the pipeline:

1. **Missing Data:** A significant amount of missing values was found across multiple columns, distributed among true `NaNs`, semantically empty strings (like "none"), and whitespace-only strings.
2. **Format Violations:** Several non-numeric strings (e.g., "one", "--2", "3.5") are present in `SeasonNumber` and `EpisodeNumber`.
3. **Invalid Dates:** Impossible dates (e.g., "0000-00-00") exist in the `AirDate` column.
4. **String Inconsistencies:** Text fields contain trailing spaces and multiple internal spaces.
5. **Duplicates:** Initial grouping indicates the presence of identical episodes, sometimes masked by formatting differences or missing season/episode numbers.

## 3) Standardization Layer

**Objective:** This stage applies systematic transformations to ensure that the dataset has a consistent format, enabling accurate comparisons and reliable cleaning. This includes unifying missing value representations, normalizing text fields, parsing numeric fields, and standardizing date formats.

### 3.1) Unify Missing Value Representations
Replace all types of "empty" values with NaN so that the rest of the pipeline can work with a single, consistent representation of missing data.

In [359]:
# Create a working copy of the dataset for standardization (non-destructive)
df_std = df_raw.copy()

In [360]:
# Function to check if a value is missing (NaN, semantically empty, or whitespace-only)
def is_missing(value):
    if pd.isna(value): return True
    return str(value).strip().lower() in SEMANTIC_EMPTY or str(value).strip() == ""

In [361]:
# Apply the function to all cells in the DataFrame, replacing any type of "missing" value with NaN
df_std = df_std.applymap(lambda x: np.nan if is_missing(x) else x)

### 3.2) Normalize text fields (SeriesName, EpisodeTitle)
Normalized means: 
- trimmed,
- collapsed spaces
- lowercased for comparison

In [362]:
# Function to clean spaces from text fields (for final output)
def clean_spaces(string):
    if pd.isna(string): return string
    return re.sub(r'\s+', ' ', str(string).strip())

In [363]:
# Function to normalize strings for comparison (trim, collapse spaces, remove accents, lowercase)
def normalize_string_for_comparison(string):
    if pd.isna(string):
        return string
    # Convert to string and trim leading/trailing spaces
    string = str(string).strip()
    # Collapse multiple spaces into a single space
    string = re.sub(r'\s+', ' ', string)
    # Remove accents and diacritics
    string = unicodedata.normalize('NFKD', string) 
    string = ''.join(c for c in string if not unicodedata.combining(c))
    # Lowercase for consistent comparison
    return string.lower()  

In [364]:
# Clean spaces in the original columns for final output
df_std['SeriesName'] = df_std['SeriesName'].apply(clean_spaces)
df_std['EpisodeTitle'] = df_std['EpisodeTitle'].apply(clean_spaces)

In [365]:
# Create normalized auxiliary columns for comparison
df_std['SeriesName_normalized'] = df_std['SeriesName'].apply(normalize_string_for_comparison)
df_std['EpisodeTitle_normalized'] = df_std['EpisodeTitle'].apply(normalize_string_for_comparison)

In [366]:
# Show records where normalization modified SeriesName or EpisodeTitle
comparison_df = df_std[(df_std['SeriesName'] != df_std['SeriesName_normalized']) | (df_std['EpisodeTitle'] != df_std['EpisodeTitle_normalized'])][['SeriesName', 'SeriesName_normalized', 'EpisodeTitle', 'EpisodeTitle_normalized']]

print(f"Number of records with modified SeriesName or EpisodeTitle after normalization: {comparison_df.shape[0]}")

# Show a preview of the normalization results (first 5)
comparison_df.head()

Number of records with modified SeriesName or EpisodeTitle after normalization: 143


,SeriesName,SeriesName_normalized,EpisodeTitle,EpisodeTitle_normalized
0,Better Call Saul,better call saul,Bingo,bingo
1,Bréaking Bäd,breaking bad,Pilot,pilot
2,Dark,dark,Lies,lies
3,Better Call Saul,better call saul,Sunk Costs,sunk costs
4,The Big Bang Theory,the big bang theory,The Hamburger Postulate,the hamburger postulate


### 3.3) Parse numeric fields
Numeric fields (SeasonNumber and EpisodeNumber) should be converted to integers where possible, with invalid values set to NaN for consistent handling in the cleaning phase.

In [367]:
# Function to parse numeric fields
def parse_int(value):
    try:
        val = int(value)
        # If the value is negative or an invalid string, set to NaN
        return val if val >= 0 else np.nan
    except:
        return np.nan

In [368]:
# Overwrite numeric columns directly
df_std['SeasonNumber'] = df_std['SeasonNumber'].apply(parse_int)
df_std['EpisodeNumber'] = df_std['EpisodeNumber'].apply(parse_int)

### 3.4) Normalize AirDate field
AirDate should be parsed into a consistent date format (YYYY-MM-DD) where possible, with invalid dates set to NaN for consistent handling in the cleaning phase.

In [369]:
# Function to parse AirDate field
def parse_date(value):
    try: 
        return datetime.strptime(str(value), "%Y-%m-%d")
    except:
        # If the value is an impossible date or an invalid string, set to NaN
        return np.nan

In [370]:
# Overwrite AirDate column directly
df_std['AirDate'] = df_std['AirDate'].apply(parse_date)

**Notes on Standardization:**
From this point forward, the _normalized fields (lowercased, no diacritics) are exclusively created and reserved for the Deduplication engine. The visually cleaned fields (with proper capitalization) are retained for the final catalog output.

## 4) Data Validation & Cleaning
**Objective:** Apply the defined cleaning rules to correct invalid values and remove records that do not meet the criteria. This stage ensures that the dataset is not only standardized but also adheres to the quality standards required for accurate deduplication and final output generation.

At this point, all values are already standardized, allowing consistent decisions.

**The following rules are applied**:
- SeriesName:
  - If missing → discard record
- SeasonNumber:
  - If missing or invalid → set to 0
- EpisodeNumber:
  - If missing or invalid → set to 0
- EpisodeTitle:
  - If missing → replace with "Untitled Episode"
- AirDate:
  - If missing or invalid → replace with "Unknown"
- Records where EpisodeNumber == 0 AND EpisodeTitle == "Untitled Episode"
  AND AirDate == "Unknown" are discarded

All corrections and removals will later be tracked for reporting.

In [371]:
# Create a working copy of the dataset for cleaning
df_clean = df_std.copy()

In [372]:
# Data quality metrics to track corrections and discards
quality_metrics = {
    "total_input": len(df_raw), # Initial total number of records in the file
    "total_output": 0,          # Will be calculated at the end
    "discarded_entries": 0,     # Accumulator for discarded records
    "corrected_entries": 0,     # Accumulator for corrected records
    "duplicates_detected": 0    # Will be calculated in section 5
}

In [373]:
# Create a mask to track which rows have been corrected (initialized as False for all)
filas_corregidas = pd.Series(False, index=df_clean.index)

In [374]:
# Function to compare original and cleaned values, returning a boolean mask of rows that were modified
def track_changes(original, cleaned):
    # Use fillna("MISSING") to ensure safe comparison in Pandas
    return original.fillna("MISSING").astype(str) != cleaned.fillna("MISSING").astype(str)

### 4.1) Remove Records with Missing SeriesName

In [375]:
# Save the initial record count for metrics
initial_record_count = len(df_clean)

# SeriesName: if missing or empty → discard the record 
# (required to identify an episode)
df_clean = df_clean[df_clean['SeriesName_normalized'].notna()]

final_record_count = len(df_clean)
discarded_series_name = initial_record_count - final_record_count
quality_metrics["discarded_entries"] += discarded_series_name
print(f"Records discarded due to missing SeriesName: {discarded_series_name}")

# Align the mask of corrected rows by removing the rows that were just discarded
filas_corregidas = filas_corregidas.loc[df_clean.index]

Records discarded due to missing SeriesName: 3


### 4.2) Apply Default Values

In [376]:
# Function to clean SeasonNumber and EpisodeNumber: if valid non-negative integer → keep, else → set to 0
def clean_number(x):
    if pd.isna(x): return 0
    try:
        num = int(x)
        return num if num >= 0 else 0
    except:
        return 0

### 4.2.1) SeasonNumber

In [377]:
# Indentify rows where SeasonNumber is missing (NaN) before cleaning
cambios_season = df_clean['SeasonNumber'].isna()

# SeasonNumber: if missing or invalid → set to 0
df_clean['SeasonNumber'] = df_clean['SeasonNumber'].apply(clean_number)

# Track changes and update metrics
corrected_count = cambios_season.sum()
filas_corregidas = filas_corregidas | cambios_season

print(f"SeasonNumber values corrected to 0: {corrected_count}")

SeasonNumber values corrected to 0: 6


### 4.2.2) EpisodeNumber

In [378]:
# Indentify rows where EpisodeNumber is missing (NaN) before cleaning
cambios_episode = df_clean['EpisodeNumber'].isna()

# EpisodeNumber: if missing or invalid → set to 0
df_clean['EpisodeNumber'] = df_clean['EpisodeNumber'].apply(clean_number)

# Track changes and update metrics
corrected_count = cambios_episode.sum()
filas_corregidas = filas_corregidas | cambios_episode

print(f"EpisodeNumber values corrected to 0: {corrected_count}")

EpisodeNumber values corrected to 0: 9


### 4.2.3) EpisodeTitle

In [379]:
# Function to clean EpisodeTitle: if missing → set to "Untitled Episode"
def clean_episode_title(x):
    return "Untitled Episode" if pd.isna(x) else x

In [380]:
# Indentify rows where EpisodeTitle is missing (NaN) before cleaning
cambios_title = df_clean['EpisodeTitle'].isna()

# EpisodeTitle: if missing → set to "Untitled Episode"
df_clean['EpisodeTitle'] = df_clean['EpisodeTitle'].apply(clean_episode_title)

# Track changes and update metrics
corrected_count = cambios_title.sum()
filas_corregidas = filas_corregidas | cambios_title

print(f"EpisodeTitle values corrected to 'Untitled Episode': {corrected_count}")

EpisodeTitle values corrected to 'Untitled Episode': 6


### 4.2.4) AirDate

In [381]:
# Function to clean AirDate: if missing or invalid → set to "Unknown"
def clean_air_date(x):
    # If the value is null (NaN or Pandas NaT), return "Unknown" directly
    if pd.isna(x):
        return "Unknown"
    
    # If the value is a valid date object, format it as a string
    if hasattr(x, 'strftime'):
        return x.strftime("%Y-%m-%d")
        
    return "Unknown"

In [382]:
# Identify rows where AirDate is missing (NaN or NaT) before cleaning
cambios_airdate = df_clean['AirDate'].isna()

# AirDate: if missing or invalid → set to "Unknown"
df_clean['AirDate'] = df_clean['AirDate'].apply(clean_air_date)

# Track changes and update metrics
corrected_count = cambios_airdate.sum()
filas_corregidas = filas_corregidas | cambios_airdate

print(f"AirDate values corrected to 'Unknown': {corrected_count}")

AirDate values corrected to 'Unknown': 12


### 4.3) Remove Records with All Key Fields Missing
When Episode Number, Episode Title and Air Date are missing (the three fields), discard the record

In [383]:
# Define the condition for discarding records with all key fields missing
discard_condition = (
    (df_clean['EpisodeNumber'] == 0) &
    (df_clean['EpisodeTitle'] == "Untitled Episode") &
    (df_clean['AirDate'] == "Unknown")
)

# Count how many records meet the discard condition
discard_count = discard_condition.sum()
quality_metrics["discarded_entries"] += discard_count

print(f"Records discarded due to all key fields missing: {discard_count}")

# Apply the discard condition to filter out the records
df_clean = df_clean[~discard_condition]

# Align the mask of corrected rows by removing the rows that were just discarded
filas_corregidas = filas_corregidas[~discard_condition]

Records discarded due to all key fields missing: 3


### 4.4) Summary of Cleaning Results

In [384]:
# Calculate total corrected entries (rows that had at least one field modified)
quality_metrics["corrected_entries"] = filas_corregidas.sum()

print(f"\n" + "="*50)
print(f"SECTION 4 SUMMARY (CLEANING & VALIDATION)")
print(f"="*50)
print(f"TOTAL CORRECTED ENTRIES (ROWS MODIFIED) : {quality_metrics['corrected_entries']}")
print(f"TOTAL DISCARDED ENTRIES SO FAR          : {quality_metrics['discarded_entries']}")
print(f"="*50)


SECTION 4 SUMMARY (CLEANING & VALIDATION)
TOTAL CORRECTED ENTRIES (ROWS MODIFIED) : 24
TOTAL DISCARDED ENTRIES SO FAR          : 6


**Notes on data validation and cleaning:**
- Non-numeric values such as "one", "--2", or "3.5" are treated as invalid.
- Although some values could theoretically be interpreted (e.g., "one" → 1),
this approach was intentionally avoided to prevent introducing assumptions
not defined in the requirements.
- Instead, all invalid numeric values are set to 0, as specified in the challenge.

## 5) Deduplication
**Objective:** Identify and remove duplicate records based on the defined rules of identity, while applying a clear strategy for selecting which record to keep when duplicates are found.

**Rules of identity for duplicates:**
Episodes must be considered duplicates when they refer to the same:
- (SeriesName_normalized, SeasonNumber, EpisodeNumber)
- (SeriesName_normalized, 0, EpisodeNumber, EpisodeTitle_normalized)
- (SeriesName_normalized, SeasonNumber, 0, EpisodeTitle_normalized)

**Note on Deduplication Strategy:**
Deduplication is applied sequentially per rule using masked filtering. By sorting the dataset by data quality first, and strictly isolating deletions to records where Season/Episode is 0, this approach safely merges missing-data entries without destroying distinct legitimate seasons or episodes.

In [385]:
# Save the initial record count for metrics
initial_dedup_count = len(df_clean)

In [386]:
# Before applying the deduplication rules, we create temporary columns to help us sort the dataset by quality (best records first).
df_clean['has_valid_date'] = df_clean['AirDate'].ne("Unknown")
df_clean['has_title'] = df_clean['EpisodeTitle'].ne("Untitled Episode")
# Combinamos Season y Episode en una sola regla como exige el PDF
df_clean['has_valid_se_and_ep'] = df_clean['SeasonNumber'].gt(0) & df_clean['EpisodeNumber'].gt(0)
# Guardamos el orden original explícito para el desempate (tie-breaker)
df_clean['original_order'] = range(len(df_clean))

In [387]:
# Sort the dataset by quality: valid date > has title > has valid season/episode > original order (first encountered)
df_clean = df_clean.sort_values(
    by=['has_valid_date', 'has_title', 'has_valid_se_and_ep', 'original_order'],
    ascending=[False, False, False, True] # True asegura que mantengamos el "first encountered"
)

In [388]:
# Remove auxiliary columns to avoid altering the catalog structure
df_clean = df_clean.drop(columns=['has_valid_date', 'has_title', 'has_valid_se_and_ep', 'original_order'])

In [389]:
# Rule 1: (SeriesName_normalized, SeasonNumber, EpisodeNumber)
duplicates_1 = df_clean.duplicated(
    subset=['SeriesName_normalized', 'SeasonNumber', 'EpisodeNumber'],
    keep=False
)
duplicates_1_df = df_clean[duplicates_1]

# Count and print the number of duplicate rows based on Rule 1
duplicates_1_count = duplicates_1_df.shape[0]
print(f"Number of duplicate rows based on Rule 1: {duplicates_1_count}")

# After sorting by quality, we keep only the first occurrence of each duplicate group based on Rule 1
df_clean = df_clean.drop_duplicates(
    subset=['SeriesName_normalized', 'SeasonNumber', 'EpisodeNumber'],
    keep='first'
)

# Show the duplicate groups based on Rule 1
duplicates_1_df

Number of duplicate rows based on Rule 1: 27


,SeriesName,SeasonNumber,EpisodeNumber,EpisodeTitle,AirDate,SeriesName_normalized,EpisodeTitle_normalized
1,Bréaking Bäd,1,1,Pilot,2008-01-20,breaking bad,pilot
2,Dark,1,2,Lies,2017-12-02,dark,lies
13,Better Call Saul,2,1,Switch,2016-02-15,better call saul,switch
14,breaking bad,1,1,Pilot,2008-01-20,breaking bad,pilot
16,Better Call Saul,1,1,Uno,2015-02-08,better call saul,uno
21,Breaking Bad,1,2,Cat's in the Bag,2008-01-27,breaking bad,cat's in the bag
26,Bëtter Cáll Saul,1,1,Uno,2015-02-08,better call saul,uno
32,The Big Bang Theory,1,1,Pilot,2007-09-24,the big bang theory,pilot
39,The Big Bang Theory,2,1,The Bad Fish Paradigm,2008-09-22,the big bang theory,the bad fish paradigm
42,Better Call Saul,1,2,Mijo,2015-02-09,better call saul,mijo


In [390]:
# Rule 2: (SeriesName_normalized, 0, EpisodeNumber, EpisodeTitle_normalized)

# Find all duplicates that share Series, Episode, and Title
all_r2_dups_mask = df_clean.duplicated(
    subset=['SeriesName_normalized', 'EpisodeNumber', 'EpisodeTitle_normalized'],
    keep=False
)

# We are only interested in those duplicates where SeasonNumber == 0
duplicates_2_mask = all_r2_dups_mask & (df_clean['SeasonNumber'] == 0)
duplicates_2_df = df_clean[duplicates_2_mask]

# Count and print the number of duplicate rows based on Rule 2
duplicates_2_count = duplicates_2_df.shape[0]
print(f"Number of duplicate rows marked for deletion based on Rule 2: {duplicates_2_count}")

# Identify which rows to delete (mark as duplicates while keeping the first valid one) and only delete if SeasonNumber is 0
to_drop_r2 = df_clean.duplicated(
    subset=['SeriesName_normalized', 'EpisodeNumber', 'EpisodeTitle_normalized'],
    keep='first'
) & (df_clean['SeasonNumber'] == 0)

df_clean = df_clean[~to_drop_r2]

# Show the duplicate groups based on Rule 2
duplicates_2_df

Number of duplicate rows marked for deletion based on Rule 2: 3


,SeriesName,SeasonNumber,EpisodeNumber,EpisodeTitle,AirDate,SeriesName_normalized,EpisodeTitle_normalized
28,The Big Bang Theory,0,10,The Loobenfeld Decay,2007-11-12,the big bang theory,the loobenfeld decay
31,Dark,0,3,Ghosts,2019-06-21,dark,ghosts
62,Better Call Saul,0,2,Cobbler,2016-02-22,better call saul,cobbler


In [391]:
# Rule 3: (SeriesName_normalized, SeasonNumber, 0, EpisodeTitle_normalized)
# Find all duplicates that share Series, Season, and Title
all_r3_dups_mask = df_clean.duplicated(
    subset=['SeriesName_normalized', 'SeasonNumber', 'EpisodeTitle_normalized'],
    keep=False
)

# We are only interested in those duplicates where EpisodeNumber == 0
duplicates_3_mask = all_r3_dups_mask & (df_clean['EpisodeNumber'] == 0)
duplicates_3_df = df_clean[duplicates_3_mask]

# Count and print the number of duplicate rows based on Rule 3
duplicates_3_count = duplicates_3_df.shape[0]
print(f"Number of duplicate rows marked for deletion based on Rule 3: {duplicates_3_count}")

# Identify which rows to delete (mark as duplicates while keeping the first valid one) and only delete if EpisodeNumber is 0
to_drop_r3 = df_clean.duplicated(
    subset=['SeriesName_normalized', 'SeasonNumber', 'EpisodeTitle_normalized'],
    keep='first'
) & (df_clean['EpisodeNumber'] == 0)

df_clean = df_clean[~to_drop_r3]

# Show the duplicate groups based on Rule 3
duplicates_3_df

Number of duplicate rows marked for deletion based on Rule 3: 3


,SeriesName,SeasonNumber,EpisodeNumber,EpisodeTitle,AirDate,SeriesName_normalized,EpisodeTitle_normalized
10,Breaking Bad,3,0,Sunset,2010-04-25,breaking bad,sunset
41,Better Call Saul,1,0,Nacho,2015-02-16,better call saul,nacho
47,Breaking Bad,2,0,Seven Thirty-Seven,2009-03-08,breaking bad,seven thirty-seven


In [392]:
# Finalize the calculation of deduplication metrics
final_dedup_count = len(df_clean)
duplicates_removed = initial_dedup_count - final_dedup_count

quality_metrics["duplicates_detected"] = duplicates_removed
quality_metrics["total_output"] = final_dedup_count

print(f"\n" + "="*50)
print(f"SECTION 5 SUMMARY (DEDUPLICATION)")
print(f"="*50)
print(f"DUPLICATES DETECTED & REMOVED : {quality_metrics['duplicates_detected']}")
print(f"FINAL CATALOG SIZE            : {quality_metrics['total_output']}")
print(f"="*50)


SECTION 5 SUMMARY (DEDUPLICATION)
DUPLICATES DETECTED & REMOVED : 21
FINAL CATALOG SIZE            : 116


## 6) Clean Dataset Generation
**Objective:** Generate the final cleaned dataset containing only the required columns for the catalog output. This dataset will be saved as `episodes_clean.csv`.

Aditionally, the final dataset will be sorted by SeriesName, SeasonNumber, and EpisodeNumber to ensure a logical and user-friendly order in the catalog.

In [393]:
# Select only the required columns for the final output
final_df = df_clean[
    ['SeriesName', 'SeasonNumber', 'EpisodeNumber', 'EpisodeTitle', 'AirDate']
]

# Sort the final dataset by SeriesName, SeasonNumber, and EpisodeNumber
final_df = final_df.sort_values(
    by=['SeriesName', 'SeasonNumber', 'EpisodeNumber']
)

# Save the final cleaned dataset to a new CSV file
final_df.to_csv("episodes_clean.csv", index=False)

In [394]:
# Final preview of the cleaned dataset
final_df.head()

,SeriesName,SeasonNumber,EpisodeNumber,EpisodeTitle,AirDate
16,Better Call Saul,1,1,Uno,2015-02-08
42,Better Call Saul,1,2,Mijo,2015-02-09
87,Better Call Saul,1,3,Nacho,2015-02-16
19,Better Call Saul,1,4,Hero,2015-02-23
9,Better Call Saul,1,5,Alpine Shepherd Boy,2015-03-02


## 7. Data Quality Report
**Objective:** Generate a comprehensive report summarizing the data quality issues found, the cleaning actions taken, and the deduplication strategy applied. This report will be saved as `report.md` and will include:
- A summary of the key data quality metrics (e.g., total records, discarded entries, corrected entries, duplicates detected).
- A detailed explanation of the deduplication strategy, including how the rules were applied and how the best record was selected in each case.
- Any insights or challenges encountered during the cleaning and deduplication process.

In [395]:
# Prepare the content of the report using the collected metrics and explanations in a markdown format
report_content = f"""# Data Quality Report

## 📊 Summary of Metrics
- **Total number of input records:** {quality_metrics['total_input']}
- **Total number of output records:** {quality_metrics['total_output']}
- **Number of discarded entries:** {quality_metrics['discarded_entries']}
- **Number of corrected entries:** {quality_metrics['corrected_entries']}
- **Number of duplicates detected:** {quality_metrics['duplicates_detected']}

## 🛠️ Deduplication Strategy Explanation
The deduplication process was designed to ensure that the most complete and accurate version of an episode is retained while strictly preventing the accidental deletion of legitimate, distinct episodes. 

The strategy was executed in two main phases:

**Phase 1: Prioritization by Data Quality**
Before identifying duplicates, the entire dataset was sorted based on a quality hierarchy. Records were prioritized using the following criteria (in order of importance):
1. Having a valid Air Date (over "Unknown").
2. Having a known Episode Title (over "Untitled Episode").
3. Having a valid Season Number (>0).
4. Having a valid Episode Number (>0).

By doing this, whenever duplicates were found, keeping the `first` occurrence guaranteed that the best available record survived.

**Phase 2: Targeted Rule Application**
Duplicates were identified using normalized text fields (lowercased, trimmed, and without diacritics) for accurate comparison. The three business rules were applied with a targeted approach:
- **Rule 1 (Exact Match):** Duplicates sharing `(SeriesName, SeasonNumber, EpisodeNumber)` were removed directly, keeping the best record.
- **Rule 2 (Missing Season):** Duplicates sharing `(SeriesName, EpisodeNumber, EpisodeTitle)` were grouped. However, to avoid merging distinct seasons that share an episode name (e.g., "Pilot"), a record was only deleted if its `SeasonNumber` was 0 (missing).
- **Rule 3 (Missing Episode):** Duplicates sharing `(SeriesName, SeasonNumber, EpisodeTitle)` were grouped. Similarly, a record was only deleted if its `EpisodeNumber` was 0 (missing).

This targeted approach successfully removed redundancies while protecting the integrity of the catalog.
"""

# Write the content to the file report.md
with open("report.md", "w", encoding="utf-8") as file:
    file.write(report_content)

print("report.md Successfully Generated")

✅ report.md successfully generated!
